In [ ]:
# 1/4 - clone CUDA backend (compiler with __shared__ + for)
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
cu=d+'/Release/cuda_backend/examples/layernorm_shared.cuda.cu'
s=open(cu).read()
print('has __shared__ =', '__shared__' in s, ' has sync =', '__syncthreads()' in s)
print('has float mean =', 'float mean' in s, ' has float rstd =', 'float rstd' in s)

In [ ]:
# 2/4 - build both LN kernels (.so)
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
for name in ['layernorm','layernorm_shared']:
    cu=ex+'/'+name+'.cuda.cu'; so=ex+'/'+name+'.so'
    cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so+' -lcudart')
    p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    print(name,'build rc=',p.returncode)
    if p.returncode!=0: print(p.stderr[:2000])

In [ ]:
# 3/4 - correctness + speed: ATen vs naive .tkv vs shared .tkv
import ctypes, torch, time
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
F32=ctypes.c_float; I32=ctypes.c_int; I64=ctypes.c_int64
def load(name):
    lib=ctypes.CDLL(ex+'/'+name+'.so')
    fn=getattr(lib,'tkv_launch_layernorm_fwd' if name=='layernorm' else 'tkv_launch_layernorm_shared')
    fn.argtypes=[I64,I64,I64,I64,I32,I32,F32,I32,I32]; fn.restype=I32
    return fn
naive=load('layernorm'); shared=load('layernorm_shared')
rows=4096; h=768
x=torch.randn(rows,h,device='cuda',dtype=torch.float32)
g=torch.randn(h,device='cuda',dtype=torch.float32); b=torch.randn(h,device='cuda',dtype=torch.float32)
out=torch.empty_like(x)
ref=torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
naive(x.data_ptr(),g.data_ptr(),b.data_ptr(),out.data_ptr(),rows,h,1e-6,rows,256)
print('naive maxerr=',(out-ref).abs().max().item())
out2=torch.empty_like(x)
shared(x.data_ptr(),g.data_ptr(),b.data_ptr(),out2.data_ptr(),rows,h,1e-6,rows,256)
print('shared maxerr=',(out2-ref).abs().max().item())
def bench(fn,o,grid):
    for _ in range(3): fn(x.data_ptr(),g.data_ptr(),b.data_ptr(),o.data_ptr(),rows,h,1e-6,grid,256)
    t0=time.perf_counter()
    for _ in range(20): fn(x.data_ptr(),g.data_ptr(),b.data_ptr(),o.data_ptr(),rows,h,1e-6,grid,256)
    return (time.perf_counter()-t0)/20*1000
def bench_aten():
    for _ in range(3): torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
    t0=time.perf_counter()
    for _ in range(20): torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
    return (time.perf_counter()-t0)/20*1000
tn=bench(naive,out,rows); ts=bench(shared,out2,rows); ta=bench_aten()
print('ATen=%.3f ms  naiveTKV=%.3f ms  sharedTKV=%.3f ms'%(ta,tn,ts))
print('shared/ATen=%.2fx  shared/naive=%.2fx'%(ta/ts,tn/ts))

In [ ]:
# 4/4 - summary
print('If sharedTKV < ATen -> TokenVector kernel wins; else ATen stays for LN.')